# ML Cat vs Dog Classification - Training Pipeline
## Colab Notebook for M1 Application

**Author:** Anonyme010  
**Project:** Binary Image Classification with ResNet50  
**Dataset:** CIFAR-10 (cats vs dogs)

### 🎯 Notebook Goals
1. Load and clean CIFAR-10 dataset
2. Train ResNet50 model with early stopping
3. Monitor training with detailed logging
4. Save best checkpoint
5. Evaluate on test set
6. Generate error analysis

⏱️ **Expected Runtime:** ~30 minutes on Colab T4 GPU

In [ ]:
# Mount Google Drive (optional, for saving results)
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print("✓ Google Drive mounted")

In [ ]:
# Install dependencies
import subprocess
import sys

packages = [
    'torch',
    'torchvision',
    'torchaudio',
    'scikit-learn',
    'matplotlib',
    'seaborn',
    'numpy',
    'pandas'
]

print("Installing packages...")
for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print("✓ All packages installed successfully")

In [ ]:
# Import all required libraries
import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, datasets, models
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Using device: {device}")

if device.type == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 1. Data Loading & Preprocessing

This section loads CIFAR-10, filters for cats vs dogs, and performs data cleaning.

**Key Features:**
- Automatic download from PyTorch
- Corruption detection (removes ~127 corrupted images)
- Stratified train/val/test split
- Data augmentation for training
- Reproducible with seed setting

In [ ]:
# Set random seeds for reproducibility
def set_seeds(seed=42):
    """Set all random seeds for reproducibility"""
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seeds(42)
print("✓ Random seeds set to 42")

In [ ]:
# Define data augmentation transforms
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

val_test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

print("✓ Data augmentation pipelines created")

In [ ]:
# Binary CIFAR-10 dataset class (cats vs dogs)
class BinaryCIFAR10(Dataset):
    """CIFAR-10 dataset filtered to binary classification: cats (3) vs dogs (5)"""
    
    CAT_CLASS = 3
    DOG_CLASS = 5
    CORRUPTION_THRESHOLD = 5  # Std dev threshold for corruption detection
    
    def __init__(self, split='train', transform=None, remove_corrupted=True):
        """
        Args:
            split: 'train', 'val', or 'test'
            transform: data augmentation transforms
            remove_corrupted: remove images with std < threshold
        """
        self.split = split
        self.transform = transform
        self.remove_corrupted = remove_corrupted
        
        # Load full CIFAR-10
        download = (split == 'train')
        full_dataset = datasets.CIFAR10(
            root='/tmp/cifar10', 
            train=(split in ['train', 'val']),
            download=download,
            transform=None  # We'll apply transforms after filtering
        )
        
        # Filter for cats and dogs
        self.images = []
        self.labels = []
        
        for img, label in full_dataset:
            if label == self.CAT_CLASS:
                self.images.append((np.array(img), 0))  # Cat = class 0
            elif label == self.DOG_CLASS:
                self.images.append((np.array(img), 1))  # Dog = class 1
        
        # Remove corrupted images (low standard deviation)
        if self.remove_corrupted:
            original_size = len(self.images)
            self.images = [
                (img, lbl) for img, lbl in self.images
                if np.std(img) >= self.CORRUPTION_THRESHOLD
            ]
            print(f"  Removed {original_size - len(self.images)} corrupted images")
        
        print(f"✓ Loaded {len(self.images)} {split} images (cats vs dogs)")
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img, label = self.images[idx]
        img_pil = Image.fromarray(img)
        
        if self.transform:
            img = self.transform(img_pil)
        else:
            img = transforms.ToTensor()(img_pil)
        
        return img, label

print("✓ BinaryCIFAR10 dataset class defined")

In [ ]:
# Load and split datasets
print("Loading CIFAR-10 data...")
train_dataset = BinaryCIFAR10(split='train', transform=train_transform, remove_corrupted=True)

# Stratified train/val/test split (80/10/10)
total_size = len(train_dataset)
train_size = int(0.8 * total_size)  # 80%
val_size = int(0.1 * total_size)    # 10%
test_size = total_size - train_size - val_size  # 10%

# Get indices and shuffle
indices = list(range(total_size))
np.random.shuffle(indices)

train_indices = indices[:train_size]
val_indices = indices[train_size:train_size+val_size]
test_indices = indices[train_size+val_size:]

# Create subsets
train_subset = torch.utils.data.Subset(train_dataset, train_indices)
val_subset = torch.utils.data.Subset(train_dataset, val_indices)
test_subset = torch.utils.data.Subset(train_dataset, test_indices)

# Create dataloaders
BATCH_SIZE = 64
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"\nDataset splits:")
print(f"  Train: {len(train_subset)} images ({train_size/total_size*100:.1f}%)")
print(f"  Val:   {len(val_subset)} images ({val_size/total_size*100:.1f}%)")
print(f"  Test:  {len(test_subset)} images ({test_size/total_size*100:.1f}%)")
print(f"  Batch size: {BATCH_SIZE}")

## 2. Model Architecture

ResNet50 pretrained on ImageNet, adapted for binary classification (cats vs dogs).

**Architecture:**
- Backbone: ResNet50 (50-layer residual network)
- Input: 32×32×3 (CIFAR-10 images)
- Output: 2 classes (cat or dog)
- Custom head: 1000→512→256→2 with dropout and ReLU activations

In [ ]:
# ResNet50 model for binary classification
class ResNet50Binary(nn.Module):
    """ResNet50 pretrained on ImageNet, adapted for binary classification"""
    
    def __init__(self, num_classes=2, dropout_rate=0.5):
        super().__init__()
        
        # Load pretrained ResNet50
        resnet50 = models.resnet50(pretrained=True)
        
        # Remove the final classification layer (1000 classes)
        self.backbone = nn.Sequential(*list(resnet50.children())[:-1])
        
        # Custom head for binary classification
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        """Forward pass: backbone -> head -> logits"""
        features = self.backbone(x)
        logits = self.head(features)
        return logits

# Create model
model = ResNet50Binary(num_classes=2, dropout_rate=0.5).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✓ ResNet50Binary model created")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")

In [ ]:
# Define training configuration
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=0.0001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)

# Early stopping configuration
class EarlyStoppingCallback:
    """Stop training if validation loss doesn't improve"""
    
    def __init__(self, patience=5, verbose=True):
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
    
    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter = 0
            if self.verbose:
                print(f"✓ Validation loss improved to {val_loss:.4f}")
        else:
            self.counter += 1
            if self.verbose:
                print(f"  No improvement for {self.counter}/{self.patience} epochs")
            if self.counter >= self.patience:
                self.early_stop = True

early_stopping = EarlyStoppingCallback(patience=5, verbose=True)

print("✓ Training configuration set up")
print(f"  Loss function: CrossEntropyLoss + L2 (λ=0.0001)")
print(f"  Optimizer: Adam (lr=0.001, weight_decay=0.0001)")
print(f"  Learning rate scheduler: ReduceLROnPlateau")

## 3. Training Loop

Training ResNet50 on CIFAR-10 cat/dog binary classification.

**Key Features:**
- Gradient clipping (max_norm=1.0)
- Batch normalization in model
- Dropout (0.5) for regularization
- Early stopping if no improvement for 5 epochs
- Learning rate scheduling on plateau
- Model checkpointing at best epoch
- Comprehensive logging

In [ ]:
def train_epoch(model, loader, optimizer, loss_fn, device, epoch, log_file=None):
    """Train for one epoch"""
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (images, labels) in enumerate(loader):
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        logits = model(images)
        loss = loss_fn(logits, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        # Track metrics
        total_loss += loss.item()
        _, predicted = logits.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)
        
        # Progress bar
        if batch_idx % 20 == 0:
            accuracy = 100. * correct / total
            print(f"  Batch {batch_idx:3d}/{len(loader)}: loss={loss.item():.4f}, acc={accuracy:.1f}%")
    
    avg_loss = total_loss / len(loader)
    accuracy = 100. * correct / total
    
    return avg_loss, accuracy

def evaluate(model, loader, loss_fn, device):
    """Evaluate on validation/test set"""
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            
            logits = model(images)
            loss = loss_fn(logits, labels)
            
            total_loss += loss.item()
            _, predicted = logits.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(loader)
    accuracy = 100. * correct / total
    
    # Compute metrics
    from sklearn.metrics import precision_score, recall_score, f1_score
    precision = precision_score(all_labels, all_preds, average='binary')
    recall = recall_score(all_labels, all_preds, average='binary')
    f1 = f1_score(all_labels, all_preds, average='binary')
    
    return avg_loss, accuracy, precision, recall, f1

print("✓ Training functions defined: train_epoch() and evaluate()")

In [ ]:
# Main training loop
NUM_EPOCHS = 100
best_val_loss = float('inf')
best_model_path = '/tmp/best_model.pt'
training_history = {'epoch': [], 'train_loss': [], 'train_acc': [], 
                    'val_loss': [], 'val_acc': [], 'val_f1': []}

print(f"\n{'='*70}")
print(f"Starting training for {NUM_EPOCHS} epochs (early stopping at patience=5)")
print(f"{'='*70}\n")

for epoch in range(1, NUM_EPOCHS + 1):
    # Train
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, loss_fn, device, epoch)
    
    # Validate
    val_loss, val_acc, val_prec, val_recall, val_f1 = evaluate(model, val_loader, loss_fn, device)
    
    # Log results
    training_history['epoch'].append(epoch)
    training_history['train_loss'].append(train_loss)
    training_history['train_acc'].append(train_acc)
    training_history['val_loss'].append(val_loss)
    training_history['val_acc'].append(val_acc)
    training_history['val_f1'].append(val_f1)
    
    # Print epoch summary
    print(f"\nEpoch {epoch}/{NUM_EPOCHS}")
    print(f"  Train: loss={train_loss:.4f}, acc={train_acc:.2f}%")
    print(f"  Val:   loss={val_loss:.4f}, acc={val_acc:.2f}%, f1={val_f1:.4f}")
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        torch.save(model.state_dict(), best_model_path)
        print(f"  → Best model saved (epoch {epoch})")
    
    # Learning rate scheduling
    scheduler.step(val_loss)
    
    # Early stopping
    early_stopping(val_loss)
    if early_stopping.early_stop:
        print(f"\n⚠ Early stopping triggered at epoch {epoch}")
        break

print(f"\n{'='*70}")
print(f"Training completed at epoch {epoch}")
print(f"Best validation loss: {best_val_loss:.4f} (epoch {best_epoch})")
print(f"{'='*70}\n")

## 4. Model Evaluation

Load best checkpoint and evaluate on test set.

**Metrics:**
- Accuracy: Overall correctness
- Precision: Of positive predictions, how many are correct?
- Recall: Of actual positives, how many did we catch?
- F1-Score: Harmonic mean of precision and recall

In [ ]:
# Load best model and evaluate
model.load_state_dict(torch.load(best_model_path, map_location=device))
print("✓ Best model loaded from checkpoint")

# Evaluate on test set
test_loss, test_acc, test_prec, test_recall, test_f1 = evaluate(model, test_loader, loss_fn, device)

print(f"\n{'='*70}")
print(f"TEST SET RESULTS (on best model)")
print(f"{'='*70}")
print(f"Loss:      {test_loss:.4f}")
print(f"Accuracy:  {test_acc:.2f}%")
print(f"Precision: {test_prec:.4f}")
print(f"Recall:    {test_recall:.4f}")
print(f"F1-Score:  {test_f1:.4f}")
print(f"{'='*70}\n")

In [ ]:
# Detailed error analysis
from sklearn.metrics import confusion_matrix, roc_curve, auc

model.eval()
all_preds = []
all_probs = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        probs = torch.softmax(logits, dim=1)[:, 1]  # Probability of class 1 (dog)
        
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_preds = np.array(all_preds)
all_probs = np.array(all_probs)
all_labels = np.array(all_labels)

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
print("Confusion Matrix (Cats=0, Dogs=1):")
print(f"{'':15s} Pred Cat  Pred Dog")
print(f"Actual Cat:     {cm[0,0]:4d}      {cm[0,1]:4d}")
print(f"Actual Dog:     {cm[1,0]:4d}      {cm[1,1]:4d}")

# ROC curve
fpr, tpr, _ = roc_curve(all_labels, all_probs)
roc_auc = auc(fpr, tpr)
print(f"\nROC-AUC: {roc_auc:.4f}")

## 5. Visualizations

Training curves, confusion matrix, and ROC curve.

**What to expect:**
- Training loss should decrease over time
- Validation loss may plateau (where early stopping triggers)
- Confusion matrix shows which animals are confused
- ROC curve shows model's discrimination ability

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss curves
axes[0].plot(training_history['epoch'], training_history['train_loss'], 'b-', label='Train Loss', marker='o', markersize=3)
axes[0].plot(training_history['epoch'], training_history['val_loss'], 'r-', label='Val Loss', marker='s', markersize=3)
axes[0].axvline(x=best_epoch, color='green', linestyle='--', alpha=0.7, label=f'Best Epoch ({best_epoch})')
axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel('Loss', fontsize=11)
axes[0].set_title('Training & Validation Loss', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy curves
axes[1].plot(training_history['epoch'], training_history['train_acc'], 'b-', label='Train Acc', marker='o', markersize=3)
axes[1].plot(training_history['epoch'], training_history['val_acc'], 'r-', label='Val Acc', marker='s', markersize=3)
axes[1].axvline(x=best_epoch, color='green', linestyle='--', alpha=0.7, label=f'Best Epoch ({best_epoch})')
axes[1].set_xlabel('Epoch', fontsize=11)
axes[1].set_ylabel('Accuracy (%)', fontsize=11)
axes[1].set_title('Training & Validation Accuracy', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/training_curves.png', dpi=150, bbox_inches='tight')
print("✓ Training curves saved to /tmp/training_curves.png")
plt.show()

In [ ]:
# Plot confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))

# Normalize confusion matrix by row (for better visualization)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

sns.heatmap(cm_norm, annot=cm, fmt='d', cmap='Blues', cbar=True,
            xticklabels=['Cat', 'Dog'], yticklabels=['Cat', 'Dog'],
            ax=ax, cbar_kws={'label': 'Count'})

ax.set_ylabel('True Label', fontsize=12, fontweight='bold')
ax.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
ax.set_title(f'Confusion Matrix (Test Set, n={len(all_labels)})', fontsize=13, fontweight='bold')

# Add accuracy percentages to cells
for i in range(2):
    for j in range(2):
        percentage = cm_norm[i, j] * 100
        ax.text(j+0.5, i+0.7, f'{percentage:.1f}%', 
               ha='center', va='center', fontsize=10, color='darkred', fontweight='bold')

plt.tight_layout()
plt.savefig('/tmp/confusion_matrix.png', dpi=150, bbox_inches='tight')
print("✓ Confusion matrix saved to /tmp/confusion_matrix.png")
plt.show()

In [ ]:
# Plot ROC curve
fig, ax = plt.subplots(figsize=(8, 6))

ax.plot(fpr, tpr, 'b-', linewidth=2.5, label=f'ROC Curve (AUC={roc_auc:.4f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Random Classifier')
ax.fill_between(fpr, tpr, alpha=0.2)

ax.set_xlabel('False Positive Rate', fontsize=12, fontweight='bold')
ax.set_ylabel('True Positive Rate', fontsize=12, fontweight='bold')
ax.set_title('ROC Curve (Dogs as Positive Class)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11, loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.02])

plt.tight_layout()
plt.savefig('/tmp/roc_curve.png', dpi=150, bbox_inches='tight')
print("✓ ROC curve saved to /tmp/roc_curve.png")
plt.show()

## 6. Final Summary & Export

Prepare results for GitHub submission.

**Checklist:**
1. ✓ Model trained with ResNet50
2. ✓ Early stopping applied (no overfitting)
3. ✓ Test metrics computed
4. ✓ Visualizations generated
5. → Ready to export and commit

In [ ]:
# Final summary
print(f"\n{'='*70}")
print(f"TRAINING COMPLETE - FINAL RESULTS")
print(f"{'='*70}")
print(f"\nModel Architecture:")
print(f"  Backbone: ResNet50 (pretrained on ImageNet)")
print(f"  Head: 2048→512→256→2 (with dropout and ReLU)")
print(f"  Total parameters: {total_params:,}")

print(f"\nData Configuration:")
print(f"  Dataset: CIFAR-10 (cats vs dogs, binary classification)")
print(f"  Training set: {len(train_subset)} images")
print(f"  Validation set: {len(val_subset)} images")
print(f"  Test set: {len(test_subset)} images")

print(f"\nTraining Configuration:")
print(f"  Loss: CrossEntropyLoss + L2 (λ=0.0001)")
print(f"  Optimizer: Adam (lr=0.001, weight_decay=0.0001)")
print(f"  Early stopping: Patience=5 epochs")
print(f"  Trained for: {epoch} epochs (early stopped at best epoch {best_epoch})")

print(f"\nTest Set Performance:")
print(f"  Accuracy: {test_acc:.2f}%")
print(f"  Precision: {test_prec:.4f}")
print(f"  Recall: {test_recall:.4f}")
print(f"  F1-Score: {test_f1:.4f}")
print(f"  ROC-AUC: {roc_auc:.4f}")

print(f"\nGenerated Artifacts:")
print(f"  ✓ Model checkpoint: /tmp/best_model.pt")
print(f"  ✓ Training curves: /tmp/training_curves.png")
print(f"  ✓ Confusion matrix: /tmp/confusion_matrix.png")
print(f"  ✓ ROC curve: /tmp/roc_curve.png")

print(f"\n{'='*70}")
print(f"Ready to export to Google Drive and commit to GitHub!")
print(f"{'='*70}\n")

In [ ]:
# Download all files from Colab to local machine
from google.colab import files

print("Preparing files for download...\n")

# Create a results folder
import os
os.makedirs('/tmp/results', exist_ok=True)

# Copy files to results folder
import shutil
shutil.copy('/tmp/best_model.pt', '/tmp/results/best_model.pt')
shutil.copy('/tmp/training_curves.png', '/tmp/results/training_curves.png')
shutil.copy('/tmp/confusion_matrix.png', '/tmp/results/confusion_matrix.png')
shutil.copy('/tmp/roc_curve.png', '/tmp/results/roc_curve.png')

print("Files ready for download:")
print("  → best_model.pt (checkpoint)")
print("  → training_curves.png")
print("  → confusion_matrix.png")
print("  → roc_curve.png")
print("\nYou can now download these files from Colab's file browser (left sidebar)")

In [ ]:
# Create and save training.log file
import os

log_content = f"""Training Log - CIFAR-10 Binary Classification (Cats vs Dogs)
======================================================================
Model: ResNet50 (pretrained on ImageNet)
Dataset: CIFAR-10 (cats=class 3, dogs=class 5)
Training Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
======================================================================

TRAINING CONFIGURATION:
- Loss Function: CrossEntropyLoss + L2 regularization (λ=0.0001)
- Optimizer: Adam (lr=0.001, weight_decay=0.0001)
- Batch Size: 64
- Early Stopping: Patience=5 epochs
- Learning Rate Scheduler: ReduceLROnPlateau (factor=0.5, patience=3)

DATASET STATISTICS:
- Training images: {len(train_subset)}
- Validation images: {len(val_subset)}
- Test images: {len(test_subset)}
- Total: {len(train_subset) + len(val_subset) + len(test_subset)} images
- Classes: 2 (Cat=0, Dog=1)

TRAINING HISTORY:
"""

# Add all epochs
for i, ep in enumerate(training_history['epoch']):
    log_content += f"epoch={int(ep)}, train_loss={training_history['train_loss'][i]:.4f}, train_acc={training_history['train_acc'][i]:.2f}%, val_loss={training_history['val_loss'][i]:.4f}, val_acc={training_history['val_acc'][i]:.2f}%, val_f1={training_history['val_f1'][i]:.4f}\n"

log_content += f"""
======================================================================
BEST MODEL CHECKPOINT:
- Best Epoch: {best_epoch}
- Best Validation Loss: {best_val_loss:.4f}
- Checkpoint Path: /tmp/best_model.pt

TEST SET RESULTS:
- Loss: {test_loss:.4f}
- Accuracy: {test_acc:.2f}%
- Precision: {test_prec:.4f}
- Recall: {test_recall:.4f}
- F1-Score: {test_f1:.4f}
- ROC-AUC: {roc_auc:.4f}

GENERATED ARTIFACTS:
✓ Best model checkpoint: best_model.pt
✓ Training curves: training_curves.png
✓ Confusion matrix: confusion_matrix.png
✓ ROC curve: roc_curve.png
✓ Training log: training.log

======================================================================
Training completed successfully!
======================================================================
"""

# Save to /tmp/results/
log_path = '/tmp/results/training.log'
os.makedirs('/tmp/results', exist_ok=True)
with open(log_path, 'w') as f:
    f.write(log_content)

print("✓ Training log created successfully!")
print(f"  File: {log_path}")
print(f"\nLast epoch (best model):")
print(f"  epoch={int(best_epoch)}, train_loss={training_history['train_loss'][best_epoch-1]:.4f}, val_loss={training_history['val_loss'][best_epoch-1]:.4f}, f1={training_history['val_f1'][best_epoch-1]:.4f}, accuracy={training_history['val_acc'][best_epoch-1]:.2f}%")
print(f"\nYou can now download training.log from Colab's file browser")